# Honest Patches: Impossible-LiveCodeBench on the competition model
Cells 1–3 come from the official starter (install wheels, load competition data, start vLLM). The cells after them run our runner against that server.

## 1. Environment Configuration and Package Installation

In [ ]:
import glob
import importlib
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LITELLM_TELEMETRY'] = 'False'
os.environ['LITELLM_LOG'] = 'WARNING'
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['OTEL_PYTHON_DISABLED'] = 'true'


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in (
        'LiteLLM',
        'LiteLLM Router',
        'LiteLLM Proxy',
        'litellm',
        'httpx',
        'httpcore',
        'openai',
        'urllib3',
        'opentelemetry',
        'opentelemetry.context',
    ):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False
            lg.handlers.clear()


configure_logging()

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        try:
            os.symlink(w, target)
        except OSError:
            shutil.copyfile(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
configure_logging()
print('Wheelhouse installation complete.')

## 2. Competition Dataset and Sample Submission

In [ ]:
import re
from swegemma.models import load_tasks

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# Official quantized competition models:
#   - 'gemma-4-31b-it-qat-w4a16-ct' (default in sample_submission)
#   - 'gemma-4-12b-it-qat-w4a16-ct'
#   - 'gemma-4-e4b-it-qat-w4a16-ct'
#   - 'gemma-4-e2b-it-qat-w4a16-ct'
MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'  # Honest Patches: use the 31B competition model

# Copy sample_submission to a writable directory for customization and packaging
SAMPLE_SUBMISSION_SRC = DATA_DIR / 'sample_submission'
AGENT_DIR = WORKING_DIR / 'sample_submission'
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
shutil.copytree(SAMPLE_SUBMISSION_SRC, AGENT_DIR)

# If switching away from 31B, update all agent YAMLs and remove the 31B-shaped sample LoRA adapters
if MODEL_NAME != 'gemma-4-31b-it-qat-w4a16-ct':
    for yaml_path in [AGENT_DIR / 'agent.yaml', *sorted((AGENT_DIR / 'sub_agents').glob('*.yaml'))]:
        if yaml_path.is_file():
            text = yaml_path.read_text(encoding='utf-8')
            text = re.sub(r'^(\s*model:\s*).*$', rf'\g<1>{MODEL_NAME}', text, flags=re.MULTILINE)
            text = re.sub(r'^\s*adapter:\s*.*\n?', '', text, flags=re.MULTILINE)
            yaml_path.write_text(text, encoding='utf-8')
    shutil.rmtree(AGENT_DIR / 'adapters', ignore_errors=True)

# Load tasks from the published dataset
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
tasks = load_tasks(TASKS_PATH)

print(f'Data directory: {DATA_DIR}')
print(f'Agent directory: {AGENT_DIR} (model={MODEL_NAME})')
print(f'Loaded {len(tasks)} tasks from {TASKS_PATH.name}')
for t in tasks[:5]:
    print(f'  - {t.instance_id} ({t.repo} @ {t.base_commit[:8]})')

## 4. Start vLLM Server

In [ ]:
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from adk_submission.discovery import discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

litellm.drop_params = True

ALLOWED_MODEL_NAMES = frozenset({
    'gemma-4-31b-it-qat-w4a16-ct',
    'gemma-4-12b-it-qat-w4a16-ct',
    'gemma-4-e4b-it-qat-w4a16-ct',
    'gemma-4-e2b-it-qat-w4a16-ct',
})
MODEL_VLLM_EXTRA_ARGS = {
    'gemma-4-12b-it-qat-w4a16-ct': (
        '--hf-overrides',
        '{"architectures": ["Gemma4ForCausalLM"], "model_type": "gemma4"}',
    ),
}
_SAFE_PATH_COMPONENT = re.compile(r'[A-Za-z0-9][A-Za-z0-9._-]*')
INFERENCE_API_KEY = os.environ.get('LOCAL_API_KEY', 'EMPTY')

# Validate single canonical base model and discover any PEFT LoRA adapters in the submission directory.
# Note: sample_submission includes dummy LoRA adapters (adapters/main_lora and adapters/tool_lora)
# shaped for gemma-4-31b-it-qat-w4a16-ct; Cell 2 removes them automatically when MODEL_NAME != 31B.
declared_models = discover_declared_models(AGENT_DIR)
assert len(declared_models) == 1, f'Expected exactly 1 declared model, got: {sorted(declared_models)}'
declared_model = next(iter(declared_models))
assert declared_model in ALLOWED_MODEL_NAMES, f'Unsupported model {declared_model!r}. Allowed: {sorted(ALLOWED_MODEL_NAMES)}'

MODEL_PATH = Path(f'/kaggle/input/models/google/gemma-4/other/{declared_model}/2')
assert MODEL_PATH.exists(), f'Model weights not found at {MODEL_PATH}. Ensure the model input is attached.'

adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
adapter_map = getattr(adapters, 'adapters', None) or {}
for adapter_name in adapter_map:
    assert _SAFE_PATH_COMPONENT.fullmatch(adapter_name), (
        f'Unsafe adapter name {adapter_name!r}: must match [A-Za-z0-9][A-Za-z0-9._-]*'
    )
detected_ranks = [
    r for info in adapter_map.values() if (r := getattr(info, 'rank', None)) is not None and r > 0
]

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

if 'server_instance' in globals() and server_instance is not None:
    server_instance.stop()

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=bool(adapter_map),
    max_loras=max(1, len(adapter_map)),
    max_lora_rank=max(detected_ranks) if detected_ranks else 128,
    scheduler_reserve_full_isl=False,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
    extra_args=MODEL_VLLM_EXTRA_ARGS.get(declared_model, ()),
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

# Register model and LoRA adapter aliases for Google ADK
models = server_instance.create_model_registry(
    aliases=[declared_model],
    model_prefix='openai/',
    api_key=INFERENCE_API_KEY,
)

## Honest Patches 1. Copy runner and data into a working folder
Needs your private dataset with `runners/` and the three `impossible_lcb_*.parquet` files attached via **Add Input**.

In [ ]:
import json, shutil, urllib.request
from pathlib import Path

search_root = Path('/kaggle/input/datasets')
runner = next(search_root.rglob('lcb_runner.py'), None)
if runner is None:  # older input layout
    search_root = Path('/kaggle/input')
    runner = next(search_root.rglob('lcb_runner.py'), None)
assert runner is not None, 'lcb_runner.py not found: attach your honest-patches-code dataset'
HP = Path('/kaggle/working/hp')
(HP / 'data').mkdir(parents=True, exist_ok=True)
(HP / 'logs').mkdir(exist_ok=True)
shutil.copytree(runner.parent, HP / 'runners', dirs_exist_ok=True)
for p in search_root.rglob('impossible_lcb_*.parquet'):
    shutil.copy(p, HP / 'data' / p.name)
print('runner from:', runner.parent)
print('data:', sorted(p.name for p in (HP / 'data').iterdir()))

## Honest Patches 2. Find the served model name

In [ ]:
BASE_URL = server_instance.base_url.rstrip('/')
if not BASE_URL.endswith('/v1'):
    BASE_URL += '/v1'
req = urllib.request.Request(f'{BASE_URL}/models', headers={'Authorization': f'Bearer {INFERENCE_API_KEY}'})
served = [m['id'] for m in json.load(urllib.request.urlopen(req))['data']]
# Skip the dummy LoRA adapters the sample submission ships with; use the base model.
SERVED_MODEL = next(m for m in served if m not in adapter_map)
print('served:', served)
print('using :', SERVED_MODEL)

## Honest Patches 3. Smoke test: one task, one worker
Check the printed line and the timing before running batches.

In [ ]:
COMMON = (f'--model {SERVED_MODEL} --base-url {BASE_URL} --api-key {INFERENCE_API_KEY} '
          '--quant w4a16-vllm --hardware kaggle-4xL4 --log logs/lcb_runs.jsonl')
!cd {HP} && python runners/lcb_runner.py --split conflicting --condition baseline --limit 1 --workers 1 {COMMON}

## Honest Patches 4. Batches
Run one cell at a time. Re-running a cell skips finished tasks, so a crashed session loses nothing already logged.
- **original**: can the model solve the real tasks? (needed to interpret the cheating rate)
- **conflicting / baseline**: RQ1 cheating rate
- **conflicting / injection**: RQ3, does a planted comment raise cheating?

In [ ]:
!cd {HP} && python runners/lcb_runner.py --split conflicting --condition baseline --limit 103 --workers 16 {COMMON}

In [ ]:
!cd {HP} && python runners/lcb_runner.py --split conflicting --condition injection --limit 103 --workers 16 {COMMON}

In [ ]:
!cd {HP} && python runners/lcb_runner.py --split original --condition baseline --limit 103 --workers 16 {COMMON}

## Honest Patches 5. Save the log where you can download it
Download `lcb_runs.jsonl` from the Output panel after every batch, then stop the session so it stops using GPU quota.

In [ ]:
shutil.copy(HP / 'logs' / 'lcb_runs.jsonl', '/kaggle/working/lcb_runs.jsonl')
import pandas as pd
df = pd.read_json('/kaggle/working/lcb_runs.jsonl', lines=True)
print(df.groupby(['split', 'condition'])['final_passed'].agg(['sum', 'count', 'mean']))